#### 📝 Load environment variables and project configurations | 🏗️ CBOT PROJECT

In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path("/home/xtallet/inari/inari-henzuru-data")
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(dotenv_path=ENV_PATH)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from app.config.settings import AzureOpenAIConfig, LangSmithConfig

cfg = AzureOpenAIConfig()
ls_cfg = LangSmithConfig()

print("Azure & LangSmith config loaded")

Azure & LangSmith config loaded


#### 📝 Print LangSmith environment variables | 🏗️ CBOT PROJECT

In [2]:
print(f"LANGSMITH_ENDPOINT: {os.getenv('LANGSMITH_ENDPOINT')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

LANGSMITH_ENDPOINT: https://eu.api.smith.langchain.com
LANGSMITH_PROJECT: ChatBot-Project-Evaluations


#### 📝 Configure LangSmith environment for tracing | 🏗️ CBOT PROJECT

In [3]:
os.environ["LANGSMITH_TRACING"] = "true" if getattr(ls_cfg, "LANGSMITH_TRACING", True) else "false"
os.environ["LANGSMITH_API_KEY"] = ls_cfg.LANGSMITH_API_KEY
os.environ["LANGSMITH_ENDPOINT"] = ls_cfg.LANGSMITH_ENDPOINT
os.environ["LANGSMITH_PROJECT"] = ls_cfg.LANGSMITH_PROJECT

print("LangSmith listo para trazar")

LangSmith listo para trazar


#### 📝 Display Azure OpenAI configuration with masked API key | 🏗️ CBOT PROJECT

In [4]:
def _mask(value: str, show: int = 4) -> str:
    if not value:
        return value
    return value[:show] + "..." + value[-show:]

print("Endpoint:", cfg.AZURE_OPENAI_API_ENDPOINT)
print("API version:", cfg.AZURE_OPENAI_API_VERSION)
print("LLM deployment:", cfg.AZURE_OPENAI_LLM_DEPLOYMENT_NAME)
print("Embedding deployment:", cfg.AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME)
print("API key:", _mask(cfg.AZURE_OPENAI_API_KEY))

Endpoint: https://inari-yomeru.openai.azure.com/
API version: 2024-12-01-preview
LLM deployment: inari-yomeru-gpt-4.1
Embedding deployment: inari-yomeru-embedding
API key: 858d...87c3


#### 📝 Initialize Azure OpenAI client | 🏗️ CBOT PROJECT

In [5]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=cfg.AZURE_OPENAI_API_KEY,
    azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
    api_version=cfg.AZURE_OPENAI_API_VERSION,
)
print("Azure OpenAI client ready")

Azure OpenAI client ready


#### 📝 Import libraries for dataset generation and evaluation | 🏗️ CBOT PROJECT

In [6]:
import json
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

#### 📝 Load real JSON dataset and convert to DataFrame | 🏗️ CBOT PROJECT

In [7]:
with open("datasets/evaluate_responses.json", "r", encoding="utf-8") as f:
    data = json.load(f)

golden_real = []
for item in data:
    golden_real.append({
        "question": item["input"],
        "answer": item["nl_output"],
        "contexts": [item["sql_response"]]
    })

df_real = pd.DataFrame(golden_real)
print(f"Real Golden Dataset: {len(df_real)} entries")

Real Golden Dataset: 26 entries


In [8]:
df_real.to_csv("kitsune_df_real.csv", index=False)

#### 📝 Print each entry from the real dataset | 🏗️ CBOT PROJECT

In [9]:
for entry in golden_real:
    print(f'entry : {entry}')

entry : {'question': 'Which policy has the highest premium in our portfolio?', 'answer': 'The policies DE2500169/R01 and DE2400169 both have the highest premium in your portfolio, which is 84,274,527,852.00.', 'contexts': ['policy_reference,status_group,premium,row_num DE2500169/R01,not_written,84274527852.0000000,1 DE2400169,written,84274527852.0000000,1']}
entry : {'question': 'Which policy has the lowest premium in our portfolio?', 'answer': 'The policy with the lowest premium in your portfolio is DE2100126, with a premium of 1.00.', 'contexts': ['policy_reference,premium DE2100126,1.0000000']}
entry : {'question': 'What is the average premium per policy in the portfolio?', 'answer': 'What is the average premium per policy in the portfolio?', 'contexts': ['policy_reference,premium_average,DE2100095,66.0000000000000000,DE2100098,1400000.000000000000,DE2100111,None,DE2100126,1.00000000000000000000,DE2100130,1.00000000000000000000,DE2100131,9.0000000000000000,DE2100200,None,DE2100201,N

#### 📝 Generate synthetic dataset using RAGAS | 🏗️ CBOT PROJECT

In [10]:
from langchain.schema import Document
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

# Convert real contexts to documents for RAGAS
docs = [Document(page_content=ctx) for entry in golden_real for ctx in entry["contexts"]]

# Split if necessary
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_documents(docs)
print(f"Chunks generados para sintéticos: {len(split_documents)}")

# Initialize testset generator with Azure OpenAI
generator_llm = LangchainLLMWrapper(
    AzureChatOpenAI(
        azure_deployment=cfg.AZURE_OPENAI_LLM_DEPLOYMENT_NAME,
        openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
        azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
        api_key=cfg.AZURE_OPENAI_API_KEY,
        temperature=0
    )
)

generator_embeddings = LangchainEmbeddingsWrapper(
    AzureOpenAIEmbeddings(
        azure_deployment=cfg.AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME,
        openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
        azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
        api_key=cfg.AZURE_OPENAI_API_KEY
    )
)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# Generate synthetic dataset
dataset_synth = generator.generate_with_langchain_docs(split_documents, testset_size=10)
df_synth = pd.DataFrame(dataset_synth)
print(f"Synthetic Golden Dataset: {len(df_synth)} entries")

Chunks generados para sintéticos: 41


Applying SummaryExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/41 [00:00<?, ?it/s]

Node 39ee260b-c60f-411e-8ca8-421b2fcb98e5 does not have a summary. Skipping filtering.
Node 02a98bfc-a1b7-40f3-a8c2-7c14fd8ece88 does not have a summary. Skipping filtering.
Node cf054490-57c5-489b-b718-afa4d8b3392c does not have a summary. Skipping filtering.
Node 366cf090-a225-47d6-a677-797d82c961b6 does not have a summary. Skipping filtering.
Node ec3c1cf5-87aa-45e9-8065-73d617bc0732 does not have a summary. Skipping filtering.
Node 3b93437d-6e5e-47d1-bb89-a1f3259aa4a8 does not have a summary. Skipping filtering.
Node d285531b-4bab-49ca-adda-1c5a88492549 does not have a summary. Skipping filtering.
Node 98506c6a-d5e7-4f21-8145-06fb2833c2fb does not have a summary. Skipping filtering.
Node 55eea9fc-7da4-4a70-b4d9-98e354fe6518 does not have a summary. Skipping filtering.
Node ead5aead-96de-413c-beed-dbc0890be1b6 does not have a summary. Skipping filtering.
Node 38414832-89a4-4d3c-aa71-22642b12105d does not have a summary. Skipping filtering.
Node 4619f127-d5cb-4a24-a673-9c59ac6b5da9 d

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/101 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

Synthetic Golden Dataset: 12 entries


#### 📝 Prepare and save synthetic dataset as EvaluationDataset | 🏗️ CBOT PROJECT

In [11]:
import pandas as pd
from ragas import EvaluationDataset

df = dataset_synth.to_pandas()
pd.set_option('display.max_colwidth', 200)
display(df)
df.to_csv("kitsune_demo_ragas_evaluations.csv", index=False)
print("✅ EvaluationDataset prepared from synthetic dataset.")

,user_input,reference_contexts,reference,synthesizer_name
0,"What is the status group and premium amount associated with the insurance policy reference DE2500169/R01, and how does it compare to the policy DE2400169 based on the provided transaction data?","[policy_reference,status_group,premium,row_num DE2500169/R01,not_written,84274527852.0000000,1 DE2400169,written,84274527852.0000000,1]","According to the provided transaction data, the insurance policy reference DE2500169/R01 has a status group of not_written and a premium amount of 84274527852.0000000. In comparison, the policy DE...",single_hop_specifc_query_synthesizer
1,What premium for DE2100126 policy?,"[policy_reference,premium DE2100126,1.0000000]",The premium for policy DE2100126 is 1.0000000.,single_hop_specifc_query_synthesizer
2,What DE2100098 mean?,"[policy_reference,premium_average,DE2100095,66.0000000000000000,DE2100098,1400000.000000000000,DE2100111,None,DE2100126,1.00000000000000000000,DE2100130,1.00000000000000000000,DE2100131,9.00000000...",DE2100098 have premium average 1400000.000000000000.,single_hop_specifc_query_synthesizer
3,What is the transaction amount for DE2300143?,"[00000000,DE2300138,30000.000000000000,DE2300139,15000.0000000000000000,DE2300140,10000.0000000000000000,DE2300141,60000.000000000000,DE2300142,50000.000000000000,DE2300143,10000.0000000000000000,...",The transaction amount for DE2300143 is 10000.0000000000000000.,single_hop_specifc_query_synthesizer
4,"Based on the available financial transactions and policy records, what patterns can be identified in the performance and completeness of facultative reinsurance policies, and how do missing or 'no...","[<1-hop>\n\n00000000,DE2300138,30000.000000000000,DE2300139,15000.0000000000000000,DE2300140,10000.0000000000000000,DE2300141,60000.000000000000,DE2300142,50000.000000000000,DE2300143,10000.000000...","Analysis of the provided financial transactions and policy records reveals several patterns regarding facultative reinsurance policies. In the <2-hop> segment, there are both 'written' and 'not_wr...",multi_hop_abstract_query_synthesizer
5,what insurance transactions got big financial transactions and written status?,"[<1-hop>\n\n00000000,DE2300138,30000.000000000000,DE2300139,15000.0000000000000000,DE2300140,10000.0000000000000000,DE2300141,60000.000000000000,DE2300142,50000.000000000000,DE2300143,10000.000000...","From the context, insurance transactions with written status and big financial transactions include: on 2024-12-01, a direct written transaction of 84839182507.0000000 with a previous value of 305...",multi_hop_abstract_query_synthesizer
6,"How you track policy status when some data missing and not written, and what problem with data completeness and integrity from these records in direct and facultative-reinsurance transactions?","[<1-hop>\n\n00:00:00+00:00,written,direct,39370000.0000000,50000.0000000,39320000.0000000,78640.00;2023-10-01 00:00:00+00:00,written,direct,200000.0000000,39370000.0000000,-39170000.0000000,-99.49...","Policy status tracking is hard when many records show not_written and missing values like None for amounts and dates, especially in direct and facultative-reinsurance transactions. For example, in...",multi_hop_abstract_query_synthesizer
7,"How much financial transactions and insurance transactions got None or not written, and what is the effect on total policy performance, like, if you look at the first context with all those DE num...","[<1-hop>\n\n5000.0000000000000000,DE2400043,50000.000000000000,DE2400044,None,DE2400046,50000.000000000000,DE2400047,2000000.000000000000,DE2400047/R01,589000000.00000000,DE2400048,455877.00000000...","In the first context, many financial transactions linked to DE numbers have None values, such as DE2400046, DE2400059, DE2400060, DE2400061, DE2400066, DE2400067, DE2400068, DE2400074, DE2400076, ...",multi_hop_abstract_query_synthesizer
8,"What are the recorded premium values for the policy references DE2100201 and D

✅ EvaluationDataset prepared from synthetic dataset.


#### 📝 Compile and initialize application graph | 🏗️ CBOT PROJECT

In [ ]:
from app.application.graph import compile_graph
from app.domain.domain import CbotState

graph = compile_graph()
print("✅ Grafo compilado correctamente")

✅ Grafo compilado correctamente


/home/xtallet/inari/inari-henzuru-data/app/application/graph.py:84: RuntimeWarning: coroutine 'compile_graph' was never awaited
  compile_graph()


#### 📝 Execute compiled graph with sample question | 🏗️ CBOT PROJECT

In [ ]:
from app.application.graph import compile_graph
from app.domain.domain import CbotState

async def ejecutar_grafo(pregunta: str):
    graph = await compile_graph()
    
    initial_state = CbotState(
        question=pregunta,
        user_id="notebook_user",
        session_id=None
    )
    
    result = await graph.ainvoke(initial_state)
    return result

import asyncio
resultado = await ejecutar_grafo("How many policies do I have ?")
print(f"Response: {resultado}")

INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: SELECT COUNT(cp.policy_reference) AS total_policies FROM get_premiums() cp;
INFO:root:Running query: SELECT COUNT(cp.policy_reference) AS total_policies FROM get_premiums() cp;
INFO:root:Query result: total_policies
260
INFO:root:Querying Agno agent for summarization
INFO:root:User question: How many policies do I have ?
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"


WARNING  MemoryDb not provided.

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: You have a total of 260 policies.


Respuesta: {'question': 'How many policies do I have ?', 'user_id': 'usuario_notebook', 'session_id': '042628a0-23ea-4128-b0ef-afa136480ce0', 'context': "question: Has the premium for policy RE2500547 changed over time? \ncot: Q: Has the premium for policy RE2500547 changed over time? A: We build a reference CTE named policy_reference that holds the policy reference user has asked about. Then, we select the policies from table _policies _policymodel related to this reference and displaying: id: policy_id, version: policy versioning id. 0 means original policy, 1 means first endorsement policy, etc, original_policy_without_endorsement_id: id of the original policy (the one that has version=0), previous_policy_of_endorsement_id: id of previous policy from which the endorsement has been created, inception_date: date where policy takes action, expiry_date: date when policy expires, status: current policy status, premium: policy amount. The SQL query is: WITH _policy_reference AS (SELECT 'R

#### 📝 Run graph on synthetic dataset and collect responses | 🏗️ CBOT PROJECT

In [ ]:
from app.application.graph import compile_graph
import asyncio

graph = await compile_graph()

for test_row in dataset_synth:
    state = {
        "question": test_row.eval_sample.user_input,
        "user_id": "ragas-eval",
        "session_id": None,
    }
    result_state = await graph.ainvoke(state)

    # Agent response
    test_row.eval_sample.response = str(result_state.get("answer", "") or "")

    # Retrieved contexts (list[str] for RAGAS)
    ctx = result_state.get("context")
    if ctx is None:
        test_row.eval_sample.retrieved_contexts = []
    elif isinstance(ctx, list):
        test_row.eval_sample.retrieved_contexts = [str(c) for c in ctx]
    else:
        test_row.eval_sample.retrieved_contexts = [str(ctx)]

    await asyncio.sleep(6)

/tmp/ipykernel_39531/1799078718.py:4: RuntimeWarning: coroutine 'compile_graph' was never awaited
  graph = await compile_graph()
INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: SELECT policy_reference, status_group, premium FROM get_premiums() WHERE policy_reference IN ('DE2500169/R01', 'DE2400169');
INFO:root:Running query: SELECT policy_reference, status_group, premium FROM get_premiums() WHERE policy_reference IN ('DE2500169/R01', 'DE2400169');
INFO:root:Query result: policy_reference,status_group,premium
DE2400169,written,84274527852.0000000
DE2500169/R01,not_written,84274527852.00000

WARNING  MemoryDb not provided.

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: The insurance policy DE2500169/R01 is in the "not_written" status group with a premium amount of 84,274,527,852.00, while DE2400169 is in the "written" status group with the same premium amount.
INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: SELECT cp.policy_reference, cp.premium FROM get_premiums() cp WHERE cp.policy_reference = 'DE2100126';
INFO:root:Running query: SELECT cp.policy_reference, cp.premium FROM get_premiums() cp WHERE cp.policy_reference

WARNING  MemoryDb not provided.

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: The premium for the DE2100126 policy is 1.0000000.
INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: SELECT * FROM policies_policymodel WHERE reference = 'DE2100098';
INFO:root:Running query: SELECT * FROM policies_policymodel WHERE reference = 'DE2100098';
INFO:root:Query result: id,status,payment_terms_type,reference,inception_date,expiry_date,created,decline_reason,decline_comments,decline_author_name,ntu_reason,ntu_comments,ntu_author_name,insured_name

...

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: The amount for DE2400082 is 100,000, and there is no amount provided for DE2500208.
INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: SELECT cp.policy_reference, cp.premium FROM get_premiums() cp WHERE cp.policy_reference IN ('DE2400137', 'DE2400227');
INFO:root:Running query: SELECT cp.policy_reference, cp.premium FROM get_premiums() cp WHERE cp.policy_reference IN ('DE2400137', 'DE2400227');
INFO:root:Query result: policy_reference,premium
DE2400137,6000

WARNING  MemoryDb not provided.

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: DE2400137 received 600,000, while DE2400227 did not receive any money.
INFO:root:Successfully connected to LanceDB and opened table: synthetic_data
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-embedding/embeddings?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:root:Generated SQL query: WITH written_policies AS (
    SELECT gp.policy_reference, pm.bound_date, pm.insured_name,
        (SELECT SUM((plm.total_layer_premium::NUMERIC * plm.share) / 100.0)
         FROM policies_policymodel pm2
         LEFT JOIN policies_limitmodel plm ON pm2.id = plm.policy_id
         WHERE pm2.reference = gp.policy_reference AND pm2.id = 

WARNING  MemoryDb not provided.

INFO:httpx:HTTP Request: POST https://api.agno.com/v1/telemetry/agent/run/create "HTTP/1.1 200 OK"
INFO:root:Generated summary from Agno: There are no written direct policy financial outcomes for 2024-09-01 and 2025-02-01.


#### 📝 Convert synthetic dataset to DataFrame and save with new columns | 🏗️ CBOT PROJECT

In [15]:
import pandas as pd
from ragas import EvaluationDataset

df = dataset_synth.to_pandas()
pd.set_option('display.max_colwidth', 200)
display(df)
df.to_csv("kitsune_demo_ragas_evaluations_new_cols.csv", index=False)
print("✅ EvaluationDataset prepared from synthetic dataset.")

,user_input,retrieved_contexts,reference_contexts,response,reference,synthesizer_name
0,"What is the status group and premium amount associated with the insurance policy reference DE2500169/R01, and how does it compare to the policy DE2400169 based on the provided transaction data?",[question: What is the premium for policy number RE2500547?\ncot: CoT: Q: What is the premium for policy number (reference) RE2500547? A: The primary data is provided through the custom PostgreSQL...,"[policy_reference,status_group,premium,row_num DE2500169/R01,not_written,84274527852.0000000,1 DE2400169,written,84274527852.0000000,1]","The insurance policy DE2500169/R01 is in the ""not_written"" status group with a premium amount of 84,274,527,852.00, while DE2400169 is in the ""written"" status group with the same premium amount.","According to the provided transaction data, the insurance policy reference DE2500169/R01 has a status group of not_written and a premium amount of 84274527852.0000000. In comparison, the policy DE...",single_hop_specifc_query_synthesizer
1,What premium for DE2100126 policy?,[question: What is the premium for policy number RE2500547?\ncot: CoT: Q: What is the premium for policy number (reference) RE2500547? A: The primary data is provided through the custom PostgreSQL...,"[policy_reference,premium DE2100126,1.0000000]",The premium for the DE2100126 policy is 1.0000000.,The premium for policy DE2100126 is 1.0000000.,single_hop_specifc_query_synthesizer
2,What DE2100098 mean?,[question: What is the premium for policy number RE2500547?\ncot: CoT: Q: What is the premium for policy number (reference) RE2500547? A: The primary data is provided through the custom PostgreSQL...,"[policy_reference,premium_average,DE2100095,66.0000000000000000,DE2100098,1400000.000000000000,DE2100111,None,DE2100126,1.00000000000000000000,DE2100130,1.00000000000000000000,DE2100131,9.00000000...","DE2100098 is the reference number for a direct, single-payment insurance policy for Qredo Ltd in the United Kingdom, with an inception date of July 1, 2021, and expiry date of March 1, 2022.",DE2100098 have premium average 1400000.000000000000.,single_hop_specifc_query_synthesizer
3,What is the transaction amount for DE2300143?,[question: What is the premium for policy number RE2500547?\ncot: CoT: Q: What is the premium for policy number (reference) RE2500547? A: The primary data is provided through the custom PostgreSQL...,"[00000000,DE2300138,30000.000000000000,DE2300139,15000.0000000000000000,DE2300140,10000.0000000000000000,DE2300141,60000.000000000000,DE2300142,50000.000000000000,DE2300143,10000.0000000000000000,...",I don't have the information to answer that question.,The transaction amount for DE2300143 is 10000.0000000000000000.,single_hop_specifc_query_synthesizer
4,"Based on the available financial transactions and policy records, what patterns can be identified in the performance and completeness of facultative reinsurance policies, and how do missing or 'no...",[question: What is the premium retention rate for renewals?\ncot: CoT: Q: What is the premium retention rate for renewals? A: From table policies_policymodel we need to compute two values first: n...,"[<1-hop>\n\n00000000,DE2300138,30000.000000000000,DE2300139,15000.0000000000000000,DE2300140,10000.0000000000000000,DE2300141,60000.000000000000,DE2300142,50000.000000000000,DE2300143,10000.000000...",I don't have the information to answer that question.,"Analysis of the provided financial transactions and policy records reveals several patterns regarding facultative reinsurance policies. In the <2-hop> segment, there are both 'written' and 'not_wr...",multi_hop_abstract_query_synthesizer
5,what insurance transactions got big financial transactions and written status?,[question: Can you show the split between the new business premium and the renewal premium?\ncot: CoT: Q: Can you show the split between the new business premium and the renewal premium? A: The pr...,"[<1-hop>\n\n00000000,DE

✅ EvaluationDataset prepared from synthetic dataset.


In [16]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import (
    LLMContextRecall,
    FactualCorrectness,
    ContextEntityRecall,
    NoiseSensitivity
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
import time

#### 📝 Evaluate synthetic dataset with custom metrics using RAGAS | 🏗️ CBOT PROJECT

In [17]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import AzureChatOpenAI
from ragas import evaluate, RunConfig, EvaluationDataset
from ragas.metrics import LLMContextRecall, FactualCorrectness, ContextEntityRecall, NoiseSensitivity

# Configurar el LLM de Azure OpenAI para evaluación
evaluator_llm = LangchainLLMWrapper(
    AzureChatOpenAI(
        azure_deployment=cfg.AZURE_OPENAI_LLM_DEPLOYMENT_NAME,
        openai_api_version=cfg.AZURE_OPENAI_API_VERSION,
        azure_endpoint=cfg.AZURE_OPENAI_API_ENDPOINT,
        api_key=cfg.AZURE_OPENAI_API_KEY,
        temperature=0
    )
)

evaluation_dataset = EvaluationDataset.from_pandas(dataset_synth.to_pandas())

# Configuración personalizada
custom_run_config = RunConfig(timeout=360)

# Ejecutar evaluación
result = evaluate(
    experiment_name='kitsune-demo-project-ragas-evaluations',
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(llm=evaluator_llm), 
        FactualCorrectness(llm=evaluator_llm), 
        ContextEntityRecall(llm=evaluator_llm), 
        NoiseSensitivity(llm=evaluator_llm)
    ],
    run_config=custom_run_config
)

result

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://inari-yomeru.openai.azure.com/openai/deployments/inari-yomeru-gpt-4.1/chat/completions?api-v

{'context_recall': 0.0238, 'factual_correctness(mode=f1)': 0.3125, 'context_entity_recall': 0.0488, 'noise_sensitivity(mode=relevant)': 0.0000}